# Track B — Colab (T4) segmented tracking

Runs the existing Track B `scripts/track_video.py` on a hosted Colab GPU. Each 15-minute segment gets its own CSV, `.meta.json`, and log in Google Drive. Completed segments are skipped on rerun; incomplete segments are rerun with `--overwrite`.

**Before running:** set `MATCH_ID`, `VIDEO_DRIVE_PATH`, half boundaries, and `HOME_DIR_1H` in the configuration cell. Use the immutable `match_id` exported by Pitchside Tagger. Inspect the first segment's team mapping before trusting `CLUSTER_0_SIDE`.

In [ ]:
# Configuration — edit these values for the match.
REPO_URL = "https://github.com/mryudistira21/football-analytics.git"
TRACK_B_BRANCH = "track-b/automatic-tracking-v1"
MATCH_ID = "REPLACE_WITH_TAGGER_MATCH_ID"

# Use a video already in Google Drive, or set VIDEO_URL to a public URL below.
VIDEO_DRIVE_PATH = "/content/drive/MyDrive/football-analysis/videos/match.mp4"
VIDEO_URL = ""  # Optional: yt-dlp URL. Leave empty to use VIDEO_DRIVE_PATH.
COPY_VIDEO_TO_RUNTIME = True  # Copy from Drive to /content for faster repeated reads.

# Home team's first-half direction in the broadcast picture: R or L.
HOME_DIR_1H = "R"
# Cluster zero is arbitrary. Verify the first preview/CSV before treating this as home.
CLUSTER_0_SIDE = "home"

# A 45-minute half becomes three 15-minute segments. Adjust boundaries to the actual file.
SEGMENT_SECONDS = 900
PERIODS = [
    {"period": 1, "start": 0, "end": 2700, "period_direction": None},
    {"period": 2, "start": 2700, "end": 5400, "period_direction": None},
]

# T4-oriented settings; preserve the project's coordinate and CSV contracts.
FRAME_STRIDE = 3
IMGSZ = 960
PITCH_IMGSZ = 960
TEAM_DEVICE = "auto"       # auto = CUDA if free VRAM >= threshold, otherwise CPU
TEAM_MIN_FREE_VRAM_GB = 4.0
TEAM_CLASSIFY_EVERY = 30
TEAM_RECLASSIFY_AFTER = 150
TEAM_SAMPLE_SECONDS = 240
TEAM_SAMPLE_MAX_CROPS = 300
TEAM_STABILITY_RATIO = 0.75


## 1. Clone Track B and install dependencies

Colab's hosted runtime uses Google's VM, not your laptop. Select a GPU runtime in **Runtime → Change runtime type → T4 GPU** (or another available GPU). The notebook does not install or replace the runtime's PyTorch build; it verifies CUDA after installation.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shutil, time

REPO_DIR = Path("/content/football-analytics")
if not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", TRACK_B_BRANCH,
        REPO_URL, str(REPO_DIR)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", TRACK_B_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", TRACK_B_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", TRACK_B_BRANCH], check=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt"),
    "git+https://github.com/roboflow/sports.git"
], check=True)

import torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. Select a GPU runtime in Colab and rerun.")
print("GPU:", torch.cuda.get_device_name(0))
subprocess.run(["nvidia-smi"], check=False)


## 2. Mount Drive and prepare persistent folders

The per-segment CSVs, metadata, logs, model weights, and fitted team-cluster state are stored in Drive. The Hugging Face cache remains on the temporary Colab disk so model loading is faster; it can be downloaded again after a runtime reset.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

if MATCH_ID == "REPLACE_WITH_TAGGER_MATCH_ID" or not MATCH_ID.strip():
    raise ValueError("Set MATCH_ID to the immutable match_id from Pitchside Tagger.")
if HOME_DIR_1H not in {"R", "L"}:
    raise ValueError("HOME_DIR_1H must be R or L.")
if SEGMENT_SECONDS <= 0:
    raise ValueError("SEGMENT_SECONDS must be positive.")

DRIVE_RUN_DIR = Path("/content/drive/MyDrive/football-analysis/track_b") / MATCH_ID
SEGMENT_DIR = DRIVE_RUN_DIR / "segments"
LOG_DIR = DRIVE_RUN_DIR / "logs"
MODEL_DIR = DRIVE_RUN_DIR / "models"
CACHE_DIR = Path("/content/hf-cache")
for folder in [DRIVE_RUN_DIR, SEGMENT_DIR, LOG_DIR, MODEL_DIR, CACHE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

os.environ["FOOTBALL_VIDEO_DIR"] = str(DRIVE_RUN_DIR)
os.environ["FOOTBALL_MODEL_DIR"] = str(MODEL_DIR)
os.environ["HF_HOME"] = str(CACHE_DIR)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(CACHE_DIR / "hub")

print("Persistent output:", DRIVE_RUN_DIR)


## 3. Get the match video and model weights

For `VIDEO_URL`, yt-dlp is optional and uses no browser cookies. Some sources may block automated downloads; in that case, place the video in Drive and leave `VIDEO_URL` empty.

In [ ]:
if VIDEO_URL.strip():
    video_target = Path(VIDEO_DRIVE_PATH)
    video_target.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        sys.executable, "-m", "yt_dlp",
        "-f", "bv*[height<=720][fps<=30]+ba/b[height<=720]/b",
        "--merge-output-format", "mp4",
        "-o", str(video_target), VIDEO_URL
    ], check=True)
else:
    video_target = Path(VIDEO_DRIVE_PATH)

if not video_target.is_file():
    raise FileNotFoundError(f"Video not found: {video_target}. Upload it to Drive or set VIDEO_URL.")

VIDEO_PATH = video_target
if COPY_VIDEO_TO_RUNTIME:
    local_video = Path("/content") / video_target.name
    if not local_video.exists() or local_video.stat().st_size != video_target.stat().st_size:
        print("Copying video from Drive to the Colab VM...")
        shutil.copy2(video_target, local_video)
    VIDEO_PATH = local_video
print("Video:", VIDEO_PATH, "size GiB:", round(VIDEO_PATH.stat().st_size / 1024**3, 3))

subprocess.run([
    sys.executable, str(REPO_DIR / "scripts" / "download_sports_models.py"),
    "--out", str(MODEL_DIR)
], check=True)
print("Models:", MODEL_DIR)


## 4. Run segments with resume support

Each invocation is a separate process. The script writes a CSV and completion metadata for each segment. The notebook skips a segment only when both files exist and the metadata says `complete: true`. On restart, track-ID offsets are reconstructed from completed CSVs. The fitted UMAP/KMeans state is shared across segments so cluster 0/1 does not get independently refitted and relabelled for each segment.

In [ ]:
import pandas as pd

TEAM_STATE_PATH = DRIVE_RUN_DIR / "team-cluster-state.joblib"
COMBINED_CSV = DRIVE_RUN_DIR / f"{MATCH_ID}_tracking.csv"
COMBINED_META = DRIVE_RUN_DIR / f"{MATCH_ID}_tracking.meta.json"

def segment_output(period: int, segment_no: int, start_s: int, end_s: int) -> Path:
    return SEGMENT_DIR / f"{MATCH_ID}_p{period}_seg{segment_no:02d}_{start_s:06d}-{end_s:06d}.csv"

def segment_is_complete(csv_path: Path, period: int, start_s: int, end_s: int) -> bool:
    meta_path = csv_path.with_suffix(".meta.json")
    if not csv_path.is_file() or not meta_path.is_file():
        return False
    try:
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        return (
            meta.get("complete") is True
            and meta.get("match_id") == MATCH_ID
            and int(meta.get("period", -1)) == period
            and meta.get("team_cluster_0_side") == CLUSTER_0_SIDE
            and int(meta.get("frame_stride", -1)) == FRAME_STRIDE
            and int(meta.get("imgsz", -1)) == IMGSZ
            and int(meta.get("pitch_imgsz", -1)) == PITCH_IMGSZ
            and meta.get("team_device_requested") == TEAM_DEVICE
            and int(meta.get("team_classify_every", -1)) == TEAM_CLASSIFY_EVERY
            and int(meta.get("team_reclassify_after", -1)) == TEAM_RECLASSIFY_AFTER
            and abs(float(meta.get("team_stability_ratio", -1)) - TEAM_STABILITY_RATIO) < 1e-9
            and abs(float(meta.get("team_min_free_vram_gb", -1)) - TEAM_MIN_FREE_VRAM_GB) < 1e-9
            and int(meta.get("team_sample_max_crops", -1)) == TEAM_SAMPLE_MAX_CROPS
            and abs(float(meta.get("team_sample_interval", -1)) - 1.5) < 1e-9
            and abs(float(meta.get("team_sample_seconds_requested", -1)) - TEAM_SAMPLE_SECONDS) < 1e-9
            and abs(float(meta.get("source_start_seconds", -1)) - start_s) < 0.01
            and abs(float(meta.get("effective_end_seconds", -1)) - end_s) < 0.01
        )
    except (ValueError, OSError, TypeError, json.JSONDecodeError):
        return False

def next_track_id_offset(completed_csvs) -> int:
    max_id = 0  # keep track_id=0 reserved for the ball
    for path in completed_csvs:
        if not path.is_file() or path.stat().st_size == 0:
            continue
        try:
            data = pd.read_csv(path, usecols=["track_id", "role"])
            people = data.loc[data["role"] != "ball", "track_id"].dropna()
            if not people.empty:
                max_id = max(max_id, int(people.max()))
        except (ValueError, OSError, pd.errors.EmptyDataError):
            continue
    return max_id + 1

def run_one_segment(period_cfg, segment_no: int, start_s: int, end_s: int, track_offset: int):
    period = int(period_cfg["period"])
    csv_path = segment_output(period, segment_no, start_s, end_s)
    if segment_is_complete(csv_path, period, start_s, end_s):
        print(f"[skip] completed {csv_path.name}")
        return csv_path

    log_path = LOG_DIR / (csv_path.stem + ".log")
    command = [
        sys.executable, str(REPO_DIR / "scripts" / "track_video.py"),
        "--input", str(VIDEO_PATH),
        "--models", str(MODEL_DIR),
        "--cache", str(CACHE_DIR),
        "--match-id", MATCH_ID,
        "--period", str(period),
        "--home-dir-1h", HOME_DIR_1H,
        "--start-seconds", str(start_s),
        "--limit-seconds", str(end_s - start_s),
        "--frame-stride", str(FRAME_STRIDE),
        "--imgsz", str(IMGSZ),
        "--pitch-imgsz", str(PITCH_IMGSZ),
        "--team-device", TEAM_DEVICE,
        "--team-min-free-vram-gb", str(TEAM_MIN_FREE_VRAM_GB),
        "--team-classify-every", str(TEAM_CLASSIFY_EVERY),
        "--team-reclassify-after", str(TEAM_RECLASSIFY_AFTER),
        "--team-stability-ratio", str(TEAM_STABILITY_RATIO),
        "--team-sample-seconds", str(TEAM_SAMPLE_SECONDS),
        "--team-sample-max-crops", str(TEAM_SAMPLE_MAX_CROPS),
        "--team-model-state", str(TEAM_STATE_PATH),
        "--cluster-0-side", CLUSTER_0_SIDE,
        "--track-id-offset", str(track_offset),
        "--output", str(csv_path),
        "--overwrite",
    ]
    if period_cfg.get("period_direction"):
        command += ["--period-direction", f"{period}:{period_cfg['period_direction']}"]

    print(f"[run] period {period}, segment {segment_no}: {start_s}s–{end_s}s")
    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.run(command, cwd=REPO_DIR, stdout=log_file,
                                 stderr=subprocess.STDOUT, text=True)
    print(f"[exit {process.returncode}] log: {log_path}")
    if process.returncode != 0:
        print(log_path.read_text(encoding="utf-8", errors="replace")[-12000:])
        raise RuntimeError(f"Segment failed: {csv_path.name}. Rerun this cell after fixing the logged issue.")
    if not segment_is_complete(csv_path, period, start_s, end_s):
        raise RuntimeError(f"Segment process exited successfully but completion metadata is missing/incomplete: {csv_path}")
    return csv_path

completed_segments = []
track_offset = 1
for period_cfg in PERIODS:
    period = int(period_cfg["period"])
    start_period = int(period_cfg["start"])
    end_period = int(period_cfg["end"])
    if end_period <= start_period:
        raise ValueError(f"Invalid period range: {period_cfg}")
    segment_no = 0
    start_s = start_period
    while start_s < end_period:
        end_s = min(end_period, start_s + SEGMENT_SECONDS)
        csv_path = segment_output(period, segment_no, start_s, end_s)
        if segment_is_complete(csv_path, period, start_s, end_s):
            print(f"[skip] completed {csv_path.name}")
        else:
            csv_path = run_one_segment(period_cfg, segment_no, start_s, end_s, track_offset)
        completed_segments.append(csv_path)
        track_offset = next_track_id_offset(completed_segments)
        segment_no += 1
        start_s = end_s

print(f"Completed segments: {len(completed_segments)}")
print("Next person track ID offset:", track_offset)


## 5. Combine completed segment CSVs

This keeps the original Track B CSV columns and values. Per-segment CSVs and metadata remain in Drive as recovery checkpoints.

In [ ]:
import csv

if not completed_segments:
    raise RuntimeError("No completed segments found.")

segment_metadata = []
row_count = 0
fieldnames = None
with COMBINED_CSV.open("w", newline="", encoding="utf-8") as combined_file:
    writer = None
    for path in completed_segments:
        with path.open("r", newline="", encoding="utf-8") as segment_file:
            reader = csv.DictReader(segment_file)
            if fieldnames is None:
                fieldnames = reader.fieldnames
                if not fieldnames:
                    raise RuntimeError(f"Missing CSV header: {path}")
                writer = csv.DictWriter(combined_file, fieldnames=fieldnames)
                writer.writeheader()
            elif reader.fieldnames != fieldnames:
                raise RuntimeError(f"Tracking CSV schema differs in {path}")
            for row in reader:
                writer.writerow(row)
                row_count += 1

        meta_path = path.with_suffix(".meta.json")
        segment_metadata.append({
            "csv": str(path),
            "metadata": str(meta_path),
            "details": json.loads(meta_path.read_text(encoding="utf-8")),
        })

combined_meta = {
    "match_id": MATCH_ID,
    "complete": True,
    "segments": segment_metadata,
    "combined_csv": str(COMBINED_CSV),
    "row_count": row_count,
    "columns": fieldnames or [],
    "pitch_geometry": "pitchside_tagger_105x68_v1",
    "team_cluster_state": str(TEAM_STATE_PATH) if TEAM_STATE_PATH.exists() else None,
    "note": "Person track IDs are unique across processed segments via track_id_offset; IDs do not imply a confirmed shirt number or cross-camera identity. Ball track_id=0 remains reserved.",
}
COMBINED_META.write_text(json.dumps(combined_meta, indent=2), encoding="utf-8")
print("Combined CSV:", COMBINED_CSV)
print("Combined metadata:", COMBINED_META)
print("Rows:", row_count)
print("Segments:", len(completed_segments))


## 6. Inspect logs and outputs

Check the first segment's video preview separately before trusting the `CLUSTER_0_SIDE` setting. The pipeline's `camera_state=closeup` still means calibration fallback, not semantic camera-shot recognition. A runtime disconnect does not invalidate segments whose `.meta.json` says `complete: true`.